# ProperTIQ — RV / boat storage site suitability (Larimer County, CO)

**Site suitability as code.** This notebook scores real land parcels for an RV &
boat-storage use, end to end, using only **open data** you bring yourself —
ProperTIQ owns the *method*, never the data.

We will:
1. Pull open layers for a small area near **Loveland, CO** (parcels, zoning, FEMA
   floodplain from county/FEMA ArcGIS; highways & competitors from OpenStreetMap).
2. Declare a **strategy** — hard filters (zoning, size, avoid floodplain) plus
   weighted scoring (near a highway, gap in competitors, room to expand).
3. Run it, see the **ranked candidates**, a transparent **per-criterion
   breakdown** (no black box), and a map.
4. **Export** the candidates (GeoJSON) and the strategy (YAML) — the same YAML the
   no-code [config app](../app/README.md) builds and consumes.


## Setup

ProperTIQ plus the demo's data-fetch helpers. On Colab, uncomment the install line.


In [1]:
# !pip install "propertiq[examples]"   # geopandas, osmnx, folium, mapclassify
import json
import urllib.request
import warnings

import geopandas as gpd
import osmnx as ox

import propertiq as pq

warnings.simplefilter("ignore")  # keep the demo output clean


## 1 · Data (open sources)

| Layer | Source |
|---|---|
| Parcels | Larimer County GIS ArcGIS (`maps1.larimer.org`) |
| Zoning | Larimer County `LC_Zoning` |
| Floodplain (SFHA) | FEMA National Flood Hazard Layer |
| Highways, competitors | OpenStreetMap via `osmnx` |

We restrict to a small **AOI bounding box** near Loveland so every pull is fast
and stays under the county API's 1,000-record page cap. Esri GeoJSON omits the CRS
block, so we tag the coordinates as WGS84 (we requested `outSR=4326`) on read.


In [2]:
# Small AOI near Loveland, CO — (west, south, east, north) in WGS84.
BBOX = (-105.10, 40.36, -105.02, 40.42)
bbox_str = ",".join(map(str, BBOX))


def esri_geojson(base_url: str, fields: str) -> gpd.GeoDataFrame:
    """Fetch a bbox-filtered ArcGIS REST layer as a WGS84 GeoDataFrame."""
    url = (
        f"{base_url}/query?where=1%3D1&geometry={bbox_str}"
        "&geometryType=esriGeometryEnvelope&inSR=4326"
        "&spatialRel=esriSpatialRelIntersects"
        f"&outFields={fields}&outSR=4326&f=geojson"
    )
    with urllib.request.urlopen(url, timeout=60) as resp:
        gj = json.load(resp)
    return gpd.GeoDataFrame.from_features(gj["features"], crs=4326)


parcels = esri_geojson(
    "https://maps1.larimer.org/arcgis/rest/services/MapServices/Parcels/MapServer/3",
    "PARCELNUM,LOCCITY,ACCTTYPE",
)
zoning = esri_geojson(
    "https://maps1.larimer.org/arcgis/rest/services/MapServices/LC_Zoning/MapServer/0",
    "ZONING_DISTRICT,ZONING_ABBRV_DISTRICT",
)
flood = esri_geojson(
    "https://hazards.fema.gov/arcgis/rest/services/public/NFHL/MapServer/28",
    "FLD_ZONE,SFHA_TF",
)

# Highways + competitor self-storage from OpenStreetMap.
roads = ox.features_from_bbox(
    BBOX, tags={"highway": ["motorway", "trunk", "primary", "secondary"]}
)
roads = roads[roads.geometry.type.isin(["LineString", "MultiLineString"])][["geometry"]]
roads = roads.reset_index(drop=True)
try:
    competitors = ox.features_from_bbox(BBOX, tags={"shop": "storage_rental"})[["geometry"]]
    competitors = competitors.reset_index(drop=True)
except Exception:  # OSM coverage of storage is sparse; degrade gracefully
    competitors = gpd.GeoDataFrame(geometry=[], crs=4326)

print(
    f"{len(parcels)} parcels · {len(zoning)} zoning polys · {len(flood)} flood polys · "
    f"{len(roads)} road segments · {len(competitors)} competitors"
)
print("(parcels may be capped at 1,000 rows — paginate for the full county.)")


1000 parcels · 125 zoning polys · 110 flood polys · 295 road segments · 9 competitors
(parcels may be capped at 1,000 rows — paginate for the full county.)


## 2 · Prepare the parcels

The Larimer parcel record carries **no area and no zoning** — both are derived:
*area* from the geometry (in a metric CRS) and *zoning* by a spatial join to the
county zoning layer. We also reduce the floodplain to the 1%-annual **Special
Flood Hazard Areas** to avoid.


In [3]:
# Area in acres, derived from geometry (no native area field).
parcels["acres"] = parcels.to_crs(5070).area / 4046.8564224

# Zoning, attached by spatial join (parcels carry no zone of their own).
joined = gpd.sjoin(
    parcels.to_crs(5070),
    zoning.to_crs(5070)[["ZONING_ABBRV_DISTRICT", "geometry"]],
    predicate="intersects",
    how="left",
)
joined = joined[~joined.index.duplicated(keep="first")]
parcels["zone"] = joined["ZONING_ABBRV_DISTRICT"].reindex(parcels.index).values

# FEMA Special Flood Hazard Areas (the 1%-annual-chance flood zones) to avoid.
SFHA = ["A", "AE", "AH", "AO", "AR", "A99", "V", "VE"]
sfha = flood[flood["FLD_ZONE"].isin(SFHA)][["geometry"]].reset_index(drop=True)

print("zones present in AOI:", parcels["zone"].value_counts(dropna=False).to_dict())
print(f"{(parcels['acres'] >= 0.5).sum()} parcels >= 0.5 acres; {len(sfha)} flood-hazard polygons")


zones present in AOI: {nan: 922, 'RR-2': 37, 'CC': 18, 'UR-2': 9, 'IL': 9, 'PD': 2, 'MZ': 2, 'MR': 1}
126 parcels >= 0.5 acres; 57 flood-hazard polygons


## 3 · The strategy

Plain-language rules for **RV / boat storage**:

**Hard filters** (must pass all)
- Zoned **commercial (`CC`)** or **light-industrial (`IL`)** — where outdoor
  storage is allowed.
- At least **0.5 acres** — room for a storage yard.
- **Not** inside the FEMA floodplain.

**Weighted scoring** (0–100, weights auto-balance to 100%)
- **Proximity to a highway** — access & visibility (weight 0.40).
- **Competitor gap** within 3 miles — find underserved demand (0.35).
- **Acreage** — bigger means room to expand (0.25).


In [4]:
rv_storage = pq.Strategy(
    name="rv_storage",
    filters=[
        pq.AttrIn("zone", ["CC", "IL"]),   # commercial / light-industrial
        pq.MinArea(acres=0.5),             # room for a storage yard
        pq.NotWithin(sfha),                # avoid the FEMA floodplain
    ],
    score=[
        pq.Proximity(to=roads, prefer="near", weight=0.40),  # access & visibility
        pq.Gap(of=competitors, within_mi=3, weight=0.35),    # underserved demand
        pq.AttrValue("acres", prefer="high", weight=0.25),   # room to expand
    ],
)
rv_storage


Strategy(filters=[AttrIn(field='zone', values=['CC', 'IL']), MinArea(acres=0.5), NotWithin(layer=                                             geometry
0   POLYGON ((-105.07018 40.38592, -105.07019 40.3...
1   POLYGON ((-105.03073 40.38349, -105.03075 40.3...
2   POLYGON ((-105.04 40.38265, -105.04007 40.3826...
3   POLYGON ((-105.04424 40.38688, -105.04417 40.3...
4   POLYGON ((-105.0956 40.38973, -105.09588 40.38...
5   POLYGON ((-105.09386 40.38926, -105.09429 40.3...
6   POLYGON ((-105.02381 40.39054, -105.024 40.390...
7   POLYGON ((-105.03075 40.38354, -105.03072 40.3...
8   POLYGON ((-105.03072 40.38358, -105.03066 40.3...
9   POLYGON ((-105.06352 40.37828, -105.06357 40.3...
10  POLYGON ((-105.0571 40.37841, -105.05747 40.37...
11  POLYGON ((-105.09622 40.39089, -105.09633 40.3...
12  POLYGON ((-105.09313 40.39254, -105.09346 40.3...
13  POLYGON ((-105.03073 40.38349, -105.0306 40.38...
14  POLYGON ((-105.03046 40.384, -105.0305 40.3841...
15  POLYGON ((-105.04646 40.38452, -105

## 4 · Run & inspect

`run` reprojects every layer to a single metric CRS, applies the filters, scores
the survivors, and ranks them. The `score_breakdown` for each parcel sums exactly
to its score — fully explainable.


In [5]:
result = rv_storage.run(parcels)
print(f"{len(result.parcels)} candidate parcels after filtering")
result.top(10)[["PARCELNUM", "zone", "acres", "score", "rank"]]


4 candidate parcels after filtering


,PARCELNUM,zone,acres,score,rank
362,9525200017,CC,4.381808,82.014266,1
367,9525200014,CC,2.844144,59.819902,2
650,9524400004,IL,4.395078,42.500000,3
704,9524228001,IL,2.685502,35.940799,4


In [6]:
# Why each top candidate scored what it did (contributions sum to the score).
result.explain().head(10).round(1)


,rank,Proximity,Gap,AttrValue,score
362,1,39.7,17.5,24.8,82.0
367,2,40.0,17.5,2.3,59.8
650,3,0.0,17.5,25.0,42.5
704,4,18.4,17.5,0.0,35.9


## 5 · Map the candidates

In [7]:
import folium

cand = result.parcels.to_crs(4326)
minx, miny, maxx, maxy = cand.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=13)
folium.GeoJson(
    cand.assign(score=cand["score"].round(1))[["score", "geometry"]],
    tooltip=folium.GeoJsonTooltip(fields=["score"], aliases=["Score:"]),
    style_function=lambda f: {
        "fillColor": "#2c7fb8", "color": "#2c7fb8", "weight": 1, "fillOpacity": 0.55
    },
).add_to(m)
m


## 6 · Export — and the no-code tie-in

The candidates export to an open format; the **strategy** exports to YAML. That
YAML is the *same format* the no-code config app produces and consumes, so a
strategy authored here runs unchanged there (and vice-versa):

```python
pq.run("rv_storage.yaml", parcels, layers={"highways": roads,
       "competitors": competitors, "fema_floodplain": sfha})
```


In [8]:
result.to_file("rv_candidates.geojson")

pq.dump_strategy(
    rv_storage,
    "rv_storage.yaml",
    layer_names={
        id(roads): "highways",
        id(competitors): "competitors",
        id(sfha): "fema_floodplain",
    },
)
print(open("rv_storage.yaml").read())


name: rv_storage
filters:
- attr_in:
    field: zone
    values:
    - CC
    - IL
- min_area:
    acres: 0.5
- not_within:
    layer: fema_floodplain
score:
- proximity:
    to: highways
    prefer: near
    weight: 0.4
- gap:
    of: competitors
    within_mi: 3
    weight: 0.35
- attr_value:
    field: acres
    prefer: high
    weight: 0.25

